# 02 — Computing forest structure products

This notebook covers the full processing pipeline from a TileDB point-cloud array to gridded raster products:

- Canopy Height Model (CHM), DTM, DSM
- Gap fraction and effective LAI
- LiDAR structural metrics
- Aboveground biomass — Næsset model and scikit-learn wrapper
- Reading results as an xarray Dataset
- Multi-temporal change detection

All outputs are written to an `ALSZarrStore` — no GeoTIFF intermediates, no mosaic step.

**Paths** — replace `/path/to/...` with your actual paths.

## Setup

In [ ]:
import alsdb
from alsdb import ALSProvider
from alsdb.storage import ALSZarrStore
alsdb.setup_logging()  # INFO-level logging to stderr

reader = ALSProvider(storage_type="local", uri="my_array")
store = ALSZarrStore("forest.zarr")  # created automatically on first write

BBOX = (308_000, 4_688_000, 310_000, 4_690_000)  # (xmin, ymin, xmax, ymax) in CRS units
YEAR = 2021

## Canopy Height Model / DTM / DSM

`compute_all` computes DTM, DSM, and CHM in a single tiled pass.

- **DTM**: Delaunay TIN interpolation by default (`dtm_method="tin"`); IDW and min-Z binning are available as alternatives.
- **HAG**: `filters.hag_delaunay` (TIN-based, consistent with DTM) with automatic `filters.hag_nn` fallback for sparse-ground tiles.
- **Ground outlier rejection**: PDAL `filters.elm` + `filters.outlier` remove misclassified returns from Class 2 before each TIN build.

A 50 m `tile_buffer` ensures ground points beyond tile edges contribute to edge triangles.

In [ ]:
from alsdb.processing.chm import compute_all

compute_all(
    provider=reader,
    store=store,
    resolution=1.0,
    bbox=BBOX,
    year=YEAR,
    tile_size=500.0,  # 500 m sub-tiles
    tile_buffer=50.0,  # 50 m overlap buffer for accurate HAG at tile edges
    n_workers=4,
    dtm_method="tin",  # "tin" (default), "idw", or "min"
)

## Gap fraction and effective LAI

Gap fraction uses the MacArthur–Wilson first-return estimator over **classified** (ground + vegetation) returns only, so unclassified or building returns do not dilute the estimate:

$$P_{gap} = \frac{N_{ground,first}}{N_{ground,first} + N_{veg,first}}$$

Effective LAI is opt-in via `lai=True` using the Beer–Lambert inversion with optional clumping correction (Jonckheere et al. 2004):

$$L_{true} = -\frac{\ln(P_{gap})}{k \cdot \Omega}$$

where $\Omega$ is the clumping index (1 = random foliage; < 1 = clumped).

In [ ]:
from alsdb.processing.gap import LAI_K_PRESETS, compute_gap_fraction

compute_gap_fraction(
    provider=reader,
    store=store,
    resolution=10.0,
    bbox=BBOX,
    year=YEAR,
    lai=True,
    k=LAI_K_PRESETS["spherical"],  # 0.5 — random leaf angles (default)
    clumping_index=0.7,  # Ω < 1 for clumped conifer canopies; 1.0 for broadleaf
    min_density=0.5,  # mask cells below 0.5 first returns m⁻²
    tile_size=500.0,
    n_workers=4,
)

# Available k presets: "spherical" (0.5), "planophile" (0.8),
#                      "erectophile" (0.35), "conifer" (0.45)
print(LAI_K_PRESETS)

## LiDAR structural metrics

`compute_metrics` writes **16 structural metrics** to the store at the chosen resolution:

| Group | Metrics |
|---|---|
| Height percentiles | `h50`, `h75`, `h95`, `hmax`, `hmean` |
| Canopy structure | `cc` (cover), `density` (pts m⁻²), `fhd` (Foliage Height Diversity), `vci` (Vegetation Complexity Index), `crr` (Canopy Relief Ratio) |
| Height strata proportions | `pv_0_2`, `pv_2_5`, `pv_5_10`, `pv_10_20`, `pv_20_40`, `pv_above40` |

The `min_density` guard masks cells below a minimum point density to suppress unreliable metrics in data-sparse areas (e.g. below dense canopy or at flight-line edges).

In [ ]:
from alsdb.processing.biomass import compute_metrics

compute_metrics(
    provider=reader,
    store=store,
    resolution=10.0,
    bbox=BBOX,
    year=YEAR,
    min_density=1.0,  # cells below 1 pt m⁻² receive NaN for all 16 metrics
    tile_size=500.0,
    n_workers=4,
)

## Aboveground biomass — Næsset (2002) model

The default allometric model is:

$$\text{AGB} = a \cdot h_{95}^{\,b} \cdot cc^{\,c}$$

Default parameters: `a=0.8, b=1.8, c=0.5` are **placeholder generic values**.

> **Always calibrate** against field inventory plots before using results scientifically. Use `calibrate_naesset` with at least 20 co-located field plots (≥ 50 recommended).

In [ ]:
import numpy as np
from alsdb.processing.biomass import calibrate_naesset, naesset_model

# --- Calibration with field inventory plots ---
# h95_plots : P95 canopy height from ALS at each plot centre (m)
# cc_plots  : canopy cover fraction (0–1) at each plot centre
# agb_plots : field-measured AGB (Mg ha⁻¹)
#
# Requires ≥ 20 valid plots; warns below 50.
#
# (a, b, c), pcov = calibrate_naesset(h95_plots, cc_plots, agb_plots, return_cov=True)
# a_std, b_std, c_std = np.sqrt(np.diag(pcov))
# print(f"a = {a:.3f} ± {a_std:.3f}  b = {b:.3f} ± {b_std:.3f}  c = {c:.3f} ± {c_std:.3f}")

# --- Apply calibrated model ---
# compute_biomass(
#     provider=reader, store=store, resolution=10.0, bbox=BBOX, year=YEAR,
#     model_fn=lambda m: naesset_model(m, a=a, b=b, c=c),
# )

In [ ]:
from alsdb.processing.biomass import compute_biomass

compute_biomass(
    provider=reader,
    store=store,
    resolution=10.0,
    bbox=BBOX,
    year=YEAR,
    tile_size=500.0,
    n_workers=4,
)

You can also pass a custom allometric function directly:

In [ ]:
def my_model(metrics):
    return 1.2 * metrics["h95"] ** 2.1 * metrics["cc"] ** 0.6


compute_biomass(
    provider=reader,
    store=store,
    resolution=10.0,
    bbox=BBOX,
    year=YEAR,
    model_fn=my_model,
)

## Aboveground biomass — scikit-learn model

`wrap_sklearn_model` wraps any trained sklearn-compatible estimator (Random Forest, Gradient Boosting, Ridge, Pipeline, …).  
It handles reshaping the per-cell metric grids into the `(n_samples, n_features)` matrix sklearn expects and masks NaN pixels automatically.

**Default feature set** (all 16 metrics from `compute_metrics`, in `_METRIC_NAMES` order):  
`h50, h75, h95, hmax, hmean, cc, density, fhd, vci, crr, pv_0_2, pv_2_5, pv_5_10, pv_10_20, pv_20_40, pv_above40`

Pass `features=[...]` to specify a subset when the model was trained on fewer columns.

In [ ]:
from sklearn.ensemble import RandomForestRegressor
from alsdb.processing.biomass import wrap_sklearn_model

# X_train must have columns in _METRIC_NAMES order (16 features by default)
rf = RandomForestRegressor(n_estimators=200, random_state=42)
rf.fit(X_train, y_agb)

compute_biomass(
    provider=reader,
    store=store,
    resolution=10.0,
    bbox=BBOX,
    year=YEAR,
    model_fn=wrap_sklearn_model(rf),
)

In [ ]:
# Use only a subset of features (must match the order used during training)
compute_biomass(
    provider=reader,
    store=store,
    resolution=10.0,
    bbox=BBOX,
    year=YEAR,
    model_fn=wrap_sklearn_model(rf, features=["h95", "cc", "density"]),
)

## Reading results as xarray

`store.to_dataset(resolution)` opens a resolution group as a CRS-aware `xarray.Dataset` (via rioxarray if available).  
The time axis corresponds to survey year.

In [ ]:
ds1m = store.to_dataset(resolution=1.0)
ds10m = store.to_dataset(resolution=10.0)

# 1 m products
chm = ds1m["chm"].sel(time=YEAR)  # Canopy Height Model (ny, nx)
dtm = ds1m["dtm"].sel(time=YEAR)  # Digital Terrain Model
dsm = ds1m["dsm"].sel(time=YEAR)  # Digital Surface Model

# 10 m structural metrics
agb = ds10m["biomass"].sel(time=YEAR)
h95 = ds10m["h95"].sel(time=YEAR)
hmax = ds10m["hmax"].sel(time=YEAR)
cc = ds10m["cc"].sel(time=YEAR)
gap = ds10m["gap"].sel(time=YEAR)
crr = ds10m["crr"].sel(time=YEAR)  # Canopy Relief Ratio
pv_5_10 = ds10m["pv_5_10"].sel(time=YEAR)  # fraction of veg returns 5–10 m

print(ds1m)
print("\n10 m variables:", list(ds10m.data_vars))
print("CRS:", ds1m.rio.crs)

## Multi-temporal change detection

Store all survey years in the same Zarr store — the time axis handles them automatically.

`compute_change` computes pixel-wise change between two years and writes three derived products:

| Variable | Description |
|---|---|
| `{var}_delta` | Absolute change (`year_to − year_from`), same units |
| `{var}_delta_pct` | Relative change (%), NaN where `\|year_from\|` < `pct_min_abs` |
| `{var}_change_flag` | +1 gain, −1 loss, 0 no significant change (uses `min_delta`) |

In [ ]:
from alsdb.processing.change import compute_change

# Ingest all survey years first
for year in [2017, 2021, 2023]:
    compute_all(
        provider=reader,
        store=store,
        resolution=1.0,
        bbox=BBOX,
        year=year,
        tile_size=500.0,
        n_workers=4,
    )

# CHM change 2017 → 2021; suppress sub-0.5 m noise
compute_change(
    store,
    "chm",
    year_from=2017,
    year_to=2021,
    resolution=1.0,
    min_delta=0.5,  # absolute threshold (m) below which change is flagged 0
    pct_min_abs=0.5,  # avoid extreme % values where base CHM < 0.5 m
)

# Read the derived products
ds = store.to_dataset(resolution=1.0)
flag = ds["chm_change_flag"].sel(time=2021)  # +1 gain / −1 loss / 0 stable
delta = ds["chm_delta"].sel(time=2021)  # absolute change (m)
delta_pct = ds["chm_delta_pct"].sel(time=2021)  # relative change (%)

print(f"Gain pixels : {int((flag == 1).sum())}")
print(f"Loss pixels : {int((flag == -1).sum())}")
print(f"Stable      : {int((flag == 0).sum())}")